## 0. Imports and Setup

In [37]:
import sys
import logging
from pathlib import Path

import pandas as pd
import pybaseball

sys.path.insert(0, '..')

from src.data.statcast_loader import (
    fetch_statcast_season,
    fetch_statcast_date_range,
    load_statcast_from_disk,
)

# Cache Statcast responses to disk so re-runs don't re-download everything.
pybaseball.cache.enable()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('notebook')

## 1. Configuration

In [38]:
# ── Date range ──────────────────────────────────────────────────────────────
START_YEAR = 2015
END_YEAR   = 2024

# ── Test mode ───────────────────────────────────────────────────────────────
# When True, pulls a single representative week instead of full seasons.
# Set to False to run the complete 2015–2024 historical pull (~1.5 hrs).
TEST_MODE       = False
TEST_START_DATE = '2023-06-01'
TEST_END_DATE   = '2023-06-07'

# ── Output paths ────────────────────────────────────────────────────────────
# Paths are relative to the project root.  VS Code's Jupyter extension sets
# the working directory to the project root (not the notebooks/ folder), so
# these do NOT use '../'.
STATCAST_DIR = Path('data/raw/statcast')
METADATA_DIR = Path('data/raw/player_metadata')

STATCAST_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Statcast output : {STATCAST_DIR.resolve()}')
print(f'Metadata output : {METADATA_DIR.resolve()}')
print(f'Test mode       : {TEST_MODE}')
if not TEST_MODE:
    print(f'Year range      : {START_YEAR} – {END_YEAR}')
    print(f'Estimated time  : ~90 min  (resume-safe — skips seasons already on disk)')

Statcast output : /Users/nateseluga/Pitcher-Injury-Risk/data/raw/statcast
Metadata output : /Users/nateseluga/Pitcher-Injury-Risk/data/raw/player_metadata
Test mode       : False
Year range      : 2015 – 2024
Estimated time  : ~90 min  (resume-safe — skips seasons already on disk)


## 2. Statcast Pitch-Level Data

Baseball Savant (via pybaseball) is the source for pitch-level Statcast data.
Every row is one pitch, with ~80 columns covering velocity, movement, spin rate,
release point, game context, and outcome.

We pull in monthly chunks — the API times out on large date ranges. A 5-second
delay between requests keeps us on the right side of Baseball Savant's rate limits.

If a season file already exists on disk, we skip it rather than re-downloading.
This makes the notebook safe to re-run after interruptions.

In [ ]:
if TEST_MODE:
    # ── Test mode: pull one short window to verify the pipeline ─────────────
    print(f'TEST MODE — pulling {TEST_START_DATE} → {TEST_END_DATE}')
    test_df = fetch_statcast_date_range(
        start_date=TEST_START_DATE,
        end_date=TEST_END_DATE,
        save=True,
    )
    print(f'\nLoaded {len(test_df):,} pitches across {test_df["game_pk"].nunique()} games')
    print(f'Columns: {test_df.shape[1]}')
    display(test_df.head(3))

else:
    # ── Full mode: pull every season, skipping files already on disk ─────────
    season_results = {}

    for year in range(START_YEAR, END_YEAR + 1):
        out_path = STATCAST_DIR / f'season_{year}.parquet'

        if out_path.exists():
            logger.info('Season %s already on disk — skipping', year)
            df = pd.read_parquet(out_path)
            season_results[year] = len(df)
            continue

        logger.info('Fetching season %s ...', year)
        df = fetch_statcast_season(season=year, save=True)
        season_results[year] = len(df)
        logger.info('Season %s: %s pitches saved', year, f'{len(df):,}')

    summary = pd.DataFrame.from_dict(
        season_results, orient='index', columns=['pitch_count']
    )
    summary.index.name = 'season'
    print('\nSeason summary:')
    display(summary)

23:42:12  INFO  Fetching season 2015 ...
23:42:12  INFO  Fetching 2015-03-01 → 2015-03-31
0it [00:00, ?it/s]
23:42:17  INFO  Fetching 2015-04-01 → 2015-04-30
  0%|          | 0/26 [00:00<?, ?it/s]/opt/homebrew/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  4%|▍         | 1/26 [00:05<02:17,  5.52s/it]/opt/homebrew/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  8%|▊         | 2/26 [00:18<03:59, 10.00s/it]/opt/homebrew/

## 3. Data Validation

Before moving forward, we check that what we collected looks right:
- Expected columns are present
- Pitch counts are plausible
- Known pitchers appear in the data
- Missing-value rates are within acceptable bounds

In [ ]:
# Load the data we just collected (test window or full season 2023)
if TEST_MODE:
    sample_df = test_df.copy()
else:
    sample_df = load_statcast_from_disk(season=2023)

print('Shape:', sample_df.shape)
print('Date range:', sample_df['game_date'].min(), '→', sample_df['game_date'].max())
print('Unique pitchers:', sample_df['pitcher'].nunique())
print('Unique games   :', sample_df['game_pk'].nunique())

Shape: (25714, 80)
Date range: 2023-06-01 00:00:00 → 2023-06-07 00:00:00
Unique pitchers: 400
Unique games   : 88


In [ ]:
# Core columns we need downstream — flag any that are missing
REQUIRED_COLS = [
    'pitcher', 'player_name', 'game_date', 'game_pk',
    'pitch_type', 'release_speed', 'release_spin_rate',
    'release_pos_x', 'release_pos_z', 'release_extension',
    'pfx_x', 'pfx_z', 'balls', 'strikes', 'inning', 'inning_topbot',
    'at_bat_number', 'pitch_number',
]

missing_cols = [c for c in REQUIRED_COLS if c not in sample_df.columns]
if missing_cols:
    print(f'WARNING — missing required columns: {missing_cols}')
else:
    print('All required columns present.')

# Missing-value rates for the columns we care about
null_rates = sample_df[REQUIRED_COLS].isnull().mean().sort_values(ascending=False)
print('\nNull rates (required cols):')
display(null_rates[null_rates > 0].to_frame('null_rate').style.format('{:.1%}'))

All required columns present.

Null rates (required cols):


,null_rate
release_spin_rate,0.6%
pitch_type,0.3%
release_extension,0.3%
release_speed,0.2%
release_pos_x,0.2%
release_pos_z,0.2%
pfx_x,0.2%
pfx_z,0.2%


In [ ]:
# Spot-check: Gerrit Cole (592789) — should always appear in a 2023 pull
GERRIT_COLE_ID = 592789

if not TEST_MODE or GERRIT_COLE_ID in sample_df['pitcher'].values:
    cole = sample_df[sample_df['pitcher'] == GERRIT_COLE_ID]
    print(f'Gerrit Cole pitches in sample: {len(cole):,}')
    print(cole[['game_date', 'pitch_type', 'release_speed', 'release_spin_rate']].head(5).to_string())
else:
    print("Cole not in test window — that's fine, he may not have pitched this week.")

# Pitch type distribution
print('\nPitch type distribution:')
print(sample_df['pitch_type'].value_counts().to_string())

Gerrit Cole pitches in sample: 66
      game_date pitch_type  release_speed  release_spin_rate
2591 2023-06-07         FF           94.0               2364
2592 2023-06-07         CH           88.6               1814
2593 2023-06-07         FF           94.0               2367
2594 2023-06-07         CH           89.5               1689
2595 2023-06-07         CH           89.3               1661

Pitch type distribution:
pitch_type
FF    8413
SL    4175
SI    3826
CH    2757
FC    2062
CU    1679
ST    1574
FS     590
KC     452
SV      71
FA      23
FO      18
SC       4
CS       3


## 4. Player Metadata

We need a roster of all pitchers with MLBAM IDs so we can:
- Join injury transactions to Statcast records
- Compute age at the time of each game
- Attach physical attributes (height, weight, throws) as model features

pybaseball's Chadwick Bureau lookup table is the most complete free source for this.
It maps MLBAM IDs to birth dates, names, and FanGraphs IDs.

In [ ]:
# Pull the full Chadwick player ID register
from pybaseball import chadwick_register

print('Downloading Chadwick Bureau register (cached after first run)...')
register = chadwick_register(save=True)
print(f'Register shape: {register.shape}')
display(register.head(3))

Register shape: (26105, 8)


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,Bradley,Jed,605152,bradj002,bradlje01,13166,2016.0,2016.0
1,Barrios,Manuel,110625,barrm002,barrima01,1000605,1997.0,1998.0
2,Martin,Frank,118336,martf102,martifr01,1008165,1897.0,1899.0


In [ ]:
# Get the set of MLBAM pitcher IDs from our Statcast pull
if TEST_MODE:
    # Use all pitchers observed in the test window
    pitcher_mlbam_ids = set(sample_df['pitcher'].dropna().astype(int))
else:
    # Load all season files and collect unique pitcher IDs
    all_pitcher_ids = set()
    for year in range(START_YEAR, END_YEAR + 1):
        path = STATCAST_DIR / f'season_{year}.parquet'
        if path.exists():
            df_yr = pd.read_parquet(path, columns=['pitcher'])
            all_pitcher_ids.update(df_yr['pitcher'].dropna().astype(int).tolist())
    pitcher_mlbam_ids = all_pitcher_ids

print(f'Unique pitchers to look up: {len(pitcher_mlbam_ids):,}')

Unique pitchers to look up: 400


In [ ]:
# Filter the register to just our pitchers
# Chadwick uses 'key_mlbam' for the MLBAM ID
pitcher_meta = register[
    register['key_mlbam'].isin(pitcher_mlbam_ids)
].copy()

# Keep the columns relevant to injury modeling
META_COLS = [
    'key_mlbam', 'key_fangraphs', 'key_bbref',
    'name_last', 'name_first',
    'birth_year', 'birth_month', 'birth_day',
]
available_meta_cols = [c for c in META_COLS if c in pitcher_meta.columns]
pitcher_meta = pitcher_meta[available_meta_cols].copy()

# Construct a proper birth_date column.
# Older versions of pybaseball's chadwick_register() omit the birth_year/month/day
# columns.  We always create birth_date so downstream code can rely on it,
# defaulting to NaT when the source columns are absent.
if all(c in pitcher_meta.columns for c in ['birth_year', 'birth_month', 'birth_day']):
    pitcher_meta['birth_date'] = pd.to_datetime(
        pitcher_meta[['birth_year', 'birth_month', 'birth_day']]
        .rename(columns={'birth_year': 'year', 'birth_month': 'month', 'birth_day': 'day'}),
        errors='coerce',
    )
else:
    pitcher_meta['birth_date'] = pd.NaT
    print('NOTE: birth_year/month/day not in register — birth_date set to NaT. '
          'Will be supplemented from MLB Stats API in a later notebook.')

pitcher_meta = pitcher_meta.rename(columns={'key_mlbam': 'player_id'})
pitcher_meta['player_name'] = (
    pitcher_meta['name_first'].fillna('') + ' ' + pitcher_meta['name_last'].fillna('')
).str.strip()

print(f'Metadata rows matched: {len(pitcher_meta):,}')
print(f'Missing birth_date   : {pitcher_meta["birth_date"].isnull().sum():,}')
display(pitcher_meta.head(5))

NOTE: birth_year/month/day not in register — birth_date set to NaT. Will be supplemented from MLB Stats API in a later notebook.
Metadata rows matched: 400
Missing birth_date   : 400


,player_id,key_fangraphs,key_bbref,name_last,name_first,birth_date,player_name
73,642239,15094,zastrro01,Zastryzny,Rob,NaT,Rob Zastryzny
191,596001,13619,junisja01,Junis,Jakob,NaT,Jakob Junis
192,593576,11804,nerishe01,Neris,Héctor,NaT,Héctor Neris
205,445926,5448,chaveje01,Chavez,Jesse,NaT,Jesse Chavez
227,608337,15474,giolilu01,Giolito,Lucas,NaT,Lucas Giolito


In [ ]:
# Save pitcher metadata
meta_path = METADATA_DIR / 'pitchers.parquet'
pitcher_meta.to_parquet(meta_path, index=False)
print(f'Saved {len(pitcher_meta):,} pitcher records → {meta_path.resolve()}')

Saved 400 pitcher records → /Users/nateseluga/Pitcher-Injury-Risk/data/raw/player_metadata/pitchers.parquet


## 5. Data Provenance Summary

Record what we collected so future runs can detect staleness.

In [ ]:
import json
from datetime import datetime, timezone

provenance = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'test_mode': TEST_MODE,
    'statcast_files': [
        {'file': str(p.name), 'rows': pd.read_parquet(p, columns=['pitcher']).shape[0]}
        for p in sorted(STATCAST_DIR.glob('*.parquet'))
    ],
    'pitcher_metadata_rows': len(pitcher_meta),
}

prov_path = Path('data/raw/provenance.json')
prov_path.write_text(json.dumps(provenance, indent=2))

print(json.dumps(provenance, indent=2))

{
  "generated_at": "2026-06-05T02:52:17.768589+00:00",
  "test_mode": true,
  "statcast_files": [
    {
      "file": "range_20230601_20230607.parquet",
      "rows": 25714
    }
  ],
  "pitcher_metadata_rows": 400
}
